# cGAN Latent-Space Evolution

This notebook builds a single 2D embedding across multiple checkpoints (epochs) and
creates GIFs showing how the latent/output space evolves over training.

It produces two animations: 1) class-labeled evolution (classes 0,1,2) and
2) class-1 soliton positions evolving across epochs (position range -0.65 → 0.65).

The notebook generates fixed `z` and condition vectors so the comparisons are deterministic.

In [15]:
# Imports and configuration
import os
from pathlib import Path
import re
import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
from matplotlib.colors import LinearSegmentedColormap
from sklearn.decomposition import PCA
import imageio
import umap
from typing import List, Optional, Dict, Tuple
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
try:
    torch.use_deterministic_algorithms(True)
except Exception:
    # Best-effort: some environments / ops may not support full determinism
    pass

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ROOT = Path.cwd()
print(f'Using device: {DEVICE}')
print(f'Workspace root: {ROOT}')

# Where to look for checkpoints (override if needed)
CHECKPOINT_DIR = ROOT
DEFAULT_EPOCHS = list(range(2, 101, 2))  # 2,4,6,...,100

Using device: cuda
Workspace root: c:\Users\sbrad\Documents\Uni\In-this-deep-together\Improve\Simulate new data\The_way


In [16]:
# Minimal Generator/loader
import torch.nn as nn
from dataclasses import dataclass
@dataclass
class CGANBundle:
    generator: nn.Module
    discriminator: nn.Module
    history: Dict
    latent_dim: int
    cond_dim: int
    embed_dim: int
    num_classes: int
    image_h: int
    image_w: int
    epoch: int

class Generator(nn.Module):
    def __init__(self, latent_dim: int, cond_dim: int, embed_dim: int, base_channels: int = 128):
        super().__init__()
        self.cond_embed = nn.Sequential(nn.Linear(cond_dim, embed_dim), nn.LeakyReLU(0.2, inplace=True))
        self.init_h = 33
        self.init_w = 41
        self.init_channels = base_channels
        self.fc = nn.Linear(latent_dim + embed_dim, base_channels * self.init_h * self.init_w)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(base_channels, base_channels // 2, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(base_channels // 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(base_channels // 2, base_channels // 4, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(base_channels // 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base_channels // 4, 1, kernel_size=3, padding=1),
            nn.Tanh(),
        )
    def forward(self, z: torch.Tensor, cond: torch.Tensor) -> torch.Tensor:
        cond_feat = self.cond_embed(cond)
        x = torch.cat([z, cond_feat], dim=1)
        x = self.fc(x)
        x = x.view(x.size(0), self.init_channels, self.init_h, self.init_w)
        return self.net(x)

class Discriminator(nn.Module):
    def __init__(self, cond_dim: int, num_classes: int, image_h: int, image_w: int, base_channels: int = 128):
        super().__init__()
        self.image_h = int(image_h)
        self.image_w = int(image_w)
        self.features = nn.Sequential(
            nn.Conv2d(2, base_channels // 4, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(base_channels // 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base_channels // 4, base_channels // 2, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(base_channels // 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base_channels // 2, base_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_channels),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.final_h = self.image_h // 4
        self.final_w = self.image_w // 4
        feat_dim = base_channels * self.final_h * self.final_w
        self.rf_head = nn.Linear(feat_dim, 1)
        self.cls_head = nn.Linear(feat_dim, num_classes)
        self.pos_head = nn.Linear(base_channels * self.final_w, 1)
        nn.init.uniform_(self.pos_head.weight, -0.01, 0.01)
        nn.init.zeros_(self.pos_head.bias)
        self.pos_feat_norm = nn.LayerNorm(base_channels * self.final_w)
    def forward(self, img: torch.Tensor, cond: torch.Tensor):
        numeric_cond = cond[:, 3:]
        cond_map = numeric_cond.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, self.image_h, self.image_w)
        cond_map = cond_map[:, 0:1, :, :]
        feat = self.features(torch.cat([img, cond_map], dim=1))
        feat_flat = feat.view(feat.size(0), -1)
        rf_logits = self.rf_head(feat_flat).squeeze(1)
        cls_logits = self.cls_head(feat_flat)
        feat_x = feat.mean(dim=2)
        feat_x_flat = self.pos_feat_norm(feat_x.view(feat_x.size(0), -1))
        pos_pred = self.pos_head(feat_x_flat).squeeze(1)
        return rf_logits, cls_logits, pos_pred

def make_condition_tensor(cond_dim: int, class_idx: int, position: float = 0.0, valid_excitation: Optional[float] = None, quality: float = 1.0, batch_size: int = 1, device: str = 'cpu', dtype: torch.dtype = torch.float32) -> torch.Tensor:
    if class_idx not in {0,1,2}:
        raise ValueError('class_idx must be 0,1,2')
    if valid_excitation is None:
        valid_excitation = 1.0 if class_idx == 1 else 0.0
    c = torch.zeros((batch_size, cond_dim), dtype=dtype, device=device)
    c[:, class_idx] = 1.0
    c[:, 3] = float(np.clip(position, -1.0, 1.0))
    c[:, 4] = 1.0 if float(valid_excitation) >= 0.5 else 0.0
    c[:, 5] = float(np.clip(quality, 0.0, 1.0))
    c[:, 3] = c[:, 3] * c[:, 4]
    return c

def load_cgan_bundle(checkpoint_path: Path, device: str = 'cpu') -> CGANBundle:
    if not checkpoint_path.exists():
        raise FileNotFoundError(f'Checkpoint does not exist: {checkpoint_path}')
    payload = torch.load(checkpoint_path, map_location=device)
    cfg = payload.get('config', {})
    latent_dim = int(cfg.get('latent_dim', cfg.get('z_dim', 100)))
    cond_dim = int(cfg.get('cond_dim', cfg.get('condition_dim', 6)))
    embed_dim = int(cfg.get('embed_dim', cfg.get('cond_embed_dim', 32)))
    num_classes = int(cfg.get('num_classes', 3))
    image_h = int(cfg.get('image_height', cfg.get('H', 264)))
    image_w = int(cfg.get('image_width', cfg.get('W', 328)))
    generator = Generator(latent_dim, cond_dim, embed_dim).to(device)
    discriminator = Discriminator(cond_dim, num_classes, image_h, image_w).to(device)

    generator.load_state_dict(payload['g_state'])
    discriminator.load_state_dict(payload['d_state'])
    
    generator.eval()
    discriminator.eval()
    return CGANBundle(generator=generator, discriminator=discriminator, history=payload.get('history', {}), epoch=int(payload.get('epoch', -1)), latent_dim=latent_dim, cond_dim=cond_dim, embed_dim=embed_dim, num_classes=num_classes, image_h=image_h, image_w=image_w)


In [17]:
# Checkpoint discovery helpers
CKPT_REGEX = re.compile(r'cgan[_-]?(\d+)')
def get_epoch_from_path(p: Path) -> Optional[int]:
    m = CKPT_REGEX.search(p.name)
    return int(m.group(1)) if m else None

def find_checkpoint_paths(root: Path, include_epochs: Optional[List[int]] = None) -> List[Path]:
    candidates: List[Tuple[int, Path]] = []
    for ext in ('*.pt', '*.pth'):
        for p in root.rglob(ext):
            epoch = get_epoch_from_path(p)
            if epoch is None:
                continue
            if include_epochs is not None and epoch not in include_epochs:
                continue
            candidates.append((epoch, p))
    candidates.sort(key=lambda x: x[0])
    return [p for _, p in candidates]

# Example: find files matching cgan_10.pt, cgan_20.pt, ...
ckpts = find_checkpoint_paths(CHECKPOINT_DIR, include_epochs=DEFAULT_EPOCHS)
print(ckpts)

[WindowsPath('c:/Users/sbrad/Documents/Uni/In-this-deep-together/Improve/Simulate new data/The_way/cgan_2.pt'), WindowsPath('c:/Users/sbrad/Documents/Uni/In-this-deep-together/Improve/Simulate new data/The_way/cgan_4.pt'), WindowsPath('c:/Users/sbrad/Documents/Uni/In-this-deep-together/Improve/Simulate new data/The_way/cgan_6.pt'), WindowsPath('c:/Users/sbrad/Documents/Uni/In-this-deep-together/Improve/Simulate new data/The_way/cgan_8.pt'), WindowsPath('c:/Users/sbrad/Documents/Uni/In-this-deep-together/Improve/Simulate new data/The_way/cgan_10.pt'), WindowsPath('c:/Users/sbrad/Documents/Uni/In-this-deep-together/Improve/Simulate new data/The_way/cgan_12.pt'), WindowsPath('c:/Users/sbrad/Documents/Uni/In-this-deep-together/Improve/Simulate new data/The_way/cgan_14.pt'), WindowsPath('c:/Users/sbrad/Documents/Uni/In-this-deep-together/Improve/Simulate new data/The_way/cgan_16.pt'), WindowsPath('c:/Users/sbrad/Documents/Uni/In-this-deep-together/Improve/Simulate new data/The_way/cgan_18.p

In [18]:
# Deterministic fixed-input builders
def make_fixed_inputs_classes(latent_dim: int, cond_dim: int, samples_per_class: int = 200, seed: int = SEED, device: str = DEVICE):
    rng = np.random.RandomState(int(seed))
    n_classes = 3
    total = samples_per_class * n_classes
    z = rng.randn(total, latent_dim).astype(np.float32)
    cond = np.zeros((total, cond_dim), dtype=np.float32)
    for cls in range(n_classes):
        start = cls * samples_per_class
        end = start + samples_per_class
        cond[start:end, cls] = 1.0
        cond[start:end, 3] = 0.0
        cond[start:end, 4] = 1.0 if cls == 1 else 0.0
        cond[start:end, 5] = 1.0
    # ensure position is only active when valid_excitation set
    cond[:, 3] = cond[:, 3] * cond[:, 4]
    labels = np.repeat(np.arange(n_classes, dtype=np.int32), samples_per_class)
    return torch.from_numpy(z).to(device), torch.from_numpy(cond).to(device), labels

def make_fixed_inputs_positions(latent_dim: int, cond_dim: int, samples_per_position: int = 24, n_positions: int = 50, pos_min: float = -0.65, pos_max: float = 0.65, seed: int = SEED, device: str = DEVICE):
    rng = np.random.RandomState(int(seed))
    positions = np.linspace(pos_min, pos_max, n_positions).astype(np.float32)
    total = samples_per_position * n_positions
    z = rng.randn(total, latent_dim).astype(np.float32)
    cond = np.zeros((total, cond_dim), dtype=np.float32)
    for i, p in enumerate(positions):
        start = i * samples_per_position
        end = start + samples_per_position
        cond[start:end, 1] = 1.0
        cond[start:end, 3] = float(p)
        cond[start:end, 4] = 1.0
        cond[start:end, 5] = 1.0
    cond[:, 3] = cond[:, 3] * cond[:, 4]
    pos_vals = np.repeat(positions, samples_per_position)
    return torch.from_numpy(z).to(device), torch.from_numpy(cond).to(device), pos_vals


In [19]:
# Generate flattened features for a set of checkpoints (batched)
@torch.no_grad()
def generate_features_for_checkpoints(ckpt_paths: List[Path], fixed_z: torch.Tensor, fixed_cond: torch.Tensor, device: str = DEVICE, batch_size: int = 64, output_dir: Optional[Path] = None):
    """
    Generate flattened image features for each checkpoint and store them as disk-backed .npy memmap files.
    Returns a dict mapping epoch -> Path to the .npy file.
    Uses a smaller default `batch_size` to reduce transient memory pressure and
    performs explicit cleanup after each batch.
    """
    import gc
    from numpy.lib.format import open_memmap

    features_by_epoch: Dict[int, Path] = {}
    use_cuda = (device == 'cuda' or (hasattr(torch, 'cuda') and torch.cuda.is_available()))
    for p in ckpt_paths:
        epoch = get_epoch_from_path(p)
        if epoch is None:
            continue
        print(f'Generating features for epoch {epoch} from {p.name}')
        bundle = load_cgan_bundle(p, device=device)
        gen = bundle.generator
        gen.eval()
        N = int(fixed_z.shape[0])
        out_dir = Path(output_dir) if output_dir is not None else p.parent
        out_dir.mkdir(parents=True, exist_ok=True)
        fname = out_dir / f'features_epoch_{epoch}.npy'

        # We'll create a memmap once we know feature dimensionality from first batch
        mem = None
        write_idx = 0
        for i in range(0, N, batch_size):
            zb = fixed_z[i:i+batch_size].to(device)
            cb = fixed_cond[i:i+batch_size].to(device)
            imgs_t = gen(zb, cb)
            arr = imgs_t.detach().cpu().numpy()[:, 0].reshape(imgs_t.shape[0], -1).astype(np.float32)
            if mem is None:
                feat_dim = arr.shape[1]
                # create .npy memmap file with proper header
                mem = open_memmap(str(fname), mode='w+', dtype='float32', shape=(N, feat_dim))
            mem[write_idx:write_idx + arr.shape[0]] = arr
            write_idx += arr.shape[0]

            # explicit cleanup to avoid accumulating large temporaries
            try:
                del imgs_t
                del arr
            except Exception:
                pass
            if use_cuda:
                try:
                    torch.cuda.empty_cache()
                except Exception:
                    pass
            gc.collect()
        # flush memmap and cleanup
        if mem is not None:
            mem.flush()
            try:
                del mem
            except Exception:
                pass
        gc.collect()
        features_by_epoch[epoch] = fname
    return features_by_epoch


In [20]:
# Build a single PCA+UMAP embedding across all epochs and render GIFs
def build_embedding_and_make_gif(features_by_epoch: Dict[int, np.ndarray], single_epoch_meta: np.ndarray, mode: str = 'classes', output_path: str = 'latent_evolution.gif', fps: int = 2, pca_dim: int = 50, pca_chunk_size: int = 1024, umap_n_neighbors: int = 10, umap_min_dist: float = 0.1, umap_n_epochs: int = 100, align_reference: bool = True, align_mode: str = 'sequential', temporal_smoothing: float = 0.12, max_step_rotation_deg: Optional[float] = 20.0, umap_sample_size: int = 5000):
    """
    Build a single 2D embedding across all epochs and save an animated GIF.
    Memory-efficient: fits UMAP on a random PCA subsample (default 5k) then transforms blocks.

    Temporal stabilization notes:
    - align_mode='sequential' aligns each epoch to the previous one (smoother animations).
    - max_step_rotation_deg limits per-frame rotation to avoid sudden flips.
    - temporal_smoothing blends each aligned frame slightly toward the previous frame.
    """
    from io import BytesIO
    from sklearn.decomposition import IncrementalPCA
    from pathlib import Path

    def _kabsch_align(ref: np.ndarray, target: np.ndarray, max_rot_deg: Optional[float] = None) -> np.ndarray:
        if ref.shape[0] < 2 or target.shape[0] < 2:
            return target
        ref_centroid = ref.mean(axis=0)
        tar_centroid = target.mean(axis=0)
        A = ref - ref_centroid
        B = target - tar_centroid
        H = B.T @ A
        U, _, Vt = np.linalg.svd(H)
        R = Vt.T @ U.T
        # Force proper rotation (det=+1) to avoid mirror reflections.
        if np.linalg.det(R) < 0:
            Vt[-1, :] *= -1
            R = Vt.T @ U.T

        # Optionally clamp rotation magnitude per frame for temporal smoothness.
        if max_rot_deg is not None:
            theta = float(np.arctan2(R[1, 0], R[0, 0]))
            max_theta = np.deg2rad(float(max_rot_deg))
            theta = float(np.clip(theta, -max_theta, max_theta))
            c, s = np.cos(theta), np.sin(theta)
            R = np.array([[c, -s], [s, c]], dtype=np.float64)

        B_rot = B @ R
        return B_rot + ref_centroid

    epochs = sorted(features_by_epoch.keys())
    blocks = []
    sizes = []
    for e in epochs:
        v = features_by_epoch[e]
        if isinstance(v, (str, Path)):
            block = np.load(str(v), mmap_mode='r')
        else:
            block = v
        blocks.append(block)
        sizes.append(block.shape[0])

    # determine effective PCA dimensionality
    feat_dim = blocks[0].shape[1]
    total_samples = sum(sizes)
    pca_dim_eff = max(2, int(min(pca_dim, feat_dim, total_samples - 1)))

    ipca = IncrementalPCA(n_components=pca_dim_eff)
    for block in blocks:
        for i in range(0, block.shape[0], pca_chunk_size):
            ipca.partial_fit(block[i:i + pca_chunk_size])

    # Transform blocks to PCA space (streaming)
    X_pca_blocks = []
    for block in blocks:
        if block.shape[0] == 0:
            X_pca_blocks.append(np.empty((0, pca_dim_eff), dtype=np.float32))
            continue
        X_pca_blocks.append(ipca.transform(block))

    # Fit UMAP on a random subset of PCA-reduced points to save memory/time
    rng = np.random.RandomState(SEED)
    total = sum(b.shape[0] for b in X_pca_blocks)
    sample_size = min(int(umap_sample_size), total)
    if sample_size < total:
        # draw sample indices across blocks
        cum = np.cumsum([0] + [b.shape[0] for b in X_pca_blocks])
        sample_indices = rng.choice(total, size=sample_size, replace=False)
        sample_rows = []
        for idx in sample_indices:
            b_i = np.searchsorted(cum, idx, side='right') - 1
            local = int(idx - cum[b_i])
            sample_rows.append(X_pca_blocks[b_i][local])
        X_sample = np.vstack(sample_rows)
        reducer = umap.UMAP(n_components=2, n_neighbors=umap_n_neighbors, min_dist=umap_min_dist, metric='euclidean', random_state=SEED, n_epochs=umap_n_epochs, init='pca')
        reducer.fit(X_sample)
        X2d_blocks = [reducer.transform(b) if b.shape[0] > 0 else np.empty((0, 2), dtype=np.float32) for b in X_pca_blocks]
    else:
        X_pca_all = np.vstack(X_pca_blocks)
        reducer = umap.UMAP(n_components=2, n_neighbors=umap_n_neighbors, min_dist=umap_min_dist, metric='euclidean', random_state=SEED, n_epochs=umap_n_epochs, init='pca')
        X2d_all = reducer.fit_transform(X_pca_all)
        X2d_blocks = []
        idx = 0
        for s in [b.shape[0] for b in X_pca_blocks]:
            if s == 0:
                X2d_blocks.append(np.empty((0, 2), dtype=np.float32))
            else:
                X2d_blocks.append(X2d_all[idx:idx + s])
            idx += s

    # Optional temporal alignment to stabilize frame-to-frame motion.
    if align_reference and len(X2d_blocks) > 1:
        mode_norm = str(align_mode).strip().lower()
        if mode_norm not in {'reference', 'sequential'}:
            mode_norm = 'sequential'

        alpha = float(np.clip(temporal_smoothing, 0.0, 0.95))
        aligned_blocks = [X2d_blocks[0].copy()]
        anchor = aligned_blocks[0]

        for i in range(1, len(X2d_blocks)):
            tgt = X2d_blocks[i]
            ref = anchor if mode_norm == 'reference' else aligned_blocks[i - 1]
            aligned = _kabsch_align(ref, tgt, max_rot_deg=max_step_rotation_deg)

            if alpha > 0.0 and aligned.shape == ref.shape:
                aligned = (1.0 - alpha) * aligned + alpha * ref

            aligned_blocks.append(aligned)

        X2d_blocks = aligned_blocks

    # concatenate 2D blocks for global limits
    X2d_all = np.vstack(X2d_blocks)

    # compute index slices
    slices = []
    idx = 0
    for s in sizes:
        slices.append((idx, idx + s))
        idx += s

    x_min, x_max = X2d_all[:, 0].min(), X2d_all[:, 0].max()
    y_min, y_max = X2d_all[:, 1].min(), X2d_all[:, 1].max()
    x_margin = (x_max - x_min) * 0.05 if (x_max - x_min) > 0 else 1.0
    y_margin = (y_max - y_min) * 0.05 if (y_max - y_min) > 0 else 1.0
    xlim = (x_min - x_margin, x_max + x_margin)
    ylim = (y_min - y_margin, y_max + y_margin)

    frames = []
    meta_single = np.asarray(single_epoch_meta)
    class_colors = {0: '#1f77b4', 1: '#9467bd', 2: '#2ca02c'}
    blue_purple = LinearSegmentedColormap.from_list('blue_purple', ['#2b83ba', '#7b3294'])

    for i_e, epoch in enumerate(epochs):
        s, e = slices[i_e]
        X2d = X2d_blocks[i_e]
        fig, ax = plt.subplots(figsize=(7.5, 6.0))
        if mode == 'classes':
            labels = meta_single
            for cls in np.unique(labels):
                idxs = labels == cls
                ax.scatter(X2d[idxs, 0], X2d[idxs, 1], s=12, alpha=0.8, color=class_colors.get(int(cls), '#888888'), label=f'class {int(cls)}')
            ax.legend(loc='best')
        elif mode == 'positions':
            pos_vals = meta_single
            sc = ax.scatter(X2d[:, 0], X2d[:, 1], c=pos_vals, cmap=blue_purple, s=14, alpha=0.85)
            cbar = fig.colorbar(sc, ax=ax)
            cbar.set_label('conditioned position')
        ax.set_title(f'Epoch {epoch}')
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.set_xlabel('UMAP-1')
        ax.set_ylabel('UMAP-2')
        ax.grid(alpha=0.2)
        plt.tight_layout()
        buf = BytesIO()
        fig.savefig(buf, format='png', bbox_inches='tight')
        buf.seek(0)
        img = imageio.imread(buf)
        if img.ndim == 3 and img.shape[2] == 4:
            img = img[..., :3]
        frames.append(img)
        plt.close(fig)

    imageio.mimsave(output_path, frames, fps=fps)
    return output_path


In [ ]:
# Execute end-to-end: find checkpoints, extract features, and build GIFs
from pathlib import Path

# Prefer the project/The_way folder but fall back to CHECKPOINT_DIR or a recursive search
candidates = [Path.cwd() / 'Improve' / 'Simulate new data' / 'The_way', Path.cwd(), CHECKPOINT_DIR]
ckpt_dir = None
for p in candidates:
    if p and p.exists():
        found = list(p.glob('cgan_*.pt')) + list(p.glob('cgan_*.pth'))
        if found:
            ckpt_dir = p
            break
if ckpt_dir is None:
    # recursive fallback from cwd
    found = list(Path.cwd().rglob('cgan_*.pt')) + list(Path.cwd().rglob('cgan_*.pth'))
    if found:
        ckpt_dir = found[0].parent

if ckpt_dir is None:
    raise FileNotFoundError('No checkpoints found (cgan_*.pt / .pth). Set CHECKPOINT_DIR or place files in the project).')

print('Using checkpoint directory:', ckpt_dir)
# Find checkpoint paths (filtered by DEFAULT_EPOCHS if present)
ckpts = find_checkpoint_paths(ckpt_dir, include_epochs=DEFAULT_EPOCHS)
if not ckpts:
    ckpts = find_checkpoint_paths(ckpt_dir)
if not ckpts:
    raise FileNotFoundError(f'No checkpoint files found under {ckpt_dir}')
print(f'Found {len(ckpts)} checkpoints: {[p.name for p in ckpts]}')

# Load first checkpoint to infer dimensions
bundle0 = load_cgan_bundle(ckpts[0], device=DEVICE)
latent_dim = int(bundle0.latent_dim)
cond_dim = int(bundle0.cond_dim)
print(f'Inferred latent_dim={latent_dim}, cond_dim={cond_dim}')

# Smoother temporal evolution settings
SMOOTH_ALIGN_KWARGS = dict(
    align_reference=True,
    align_mode='sequential',        # align each frame to the previous frame
    temporal_smoothing=0.12,        # blend a bit toward previous frame
    max_step_rotation_deg=20.0,     # avoid sudden large rotations
    umap_n_neighbors=15,            # closer to analysis notebook defaults
)

# --- Classes GIF ---
samples_per_class = 200
z_cls, cond_cls, labels_cls = make_fixed_inputs_classes(latent_dim, cond_dim, samples_per_class=samples_per_class, seed=SEED, device=DEVICE)
print('Generating features for class-fixed inputs (this may take a while)')
feats_cls = generate_features_for_checkpoints(ckpts, z_cls, cond_cls, device=DEVICE, batch_size=128)
out_gif_classes = ckpt_dir / 'latent_classes_evolution.gif'
print('Building class evolution GIF...')
build_embedding_and_make_gif(feats_cls, labels_cls, mode='classes', output_path=str(out_gif_classes), fps=2, pca_dim=100, **SMOOTH_ALIGN_KWARGS)
print('Saved classes GIF to', out_gif_classes)

# --- Positions GIF (class 1) ---
samples_per_position = 24
n_positions = 100
z_pos, cond_pos, pos_vals = make_fixed_inputs_positions(latent_dim, cond_dim, samples_per_position=samples_per_position, n_positions=n_positions, pos_min=-0.65, pos_max=0.65, seed=SEED, device=DEVICE)
print('Generating features for position-fixed inputs (this may take a while)')
feats_pos = generate_features_for_checkpoints(ckpts, z_pos, cond_pos, device=DEVICE, batch_size=128)
out_gif_pos = ckpt_dir / 'latent_positions_evolution.gif'
print('Building position evolution GIF...')
build_embedding_and_make_gif(feats_pos, pos_vals, mode='positions', output_path=str(out_gif_pos), fps=2, pca_dim=50, **SMOOTH_ALIGN_KWARGS)
print('Saved positions GIF to', out_gif_pos)


Using checkpoint directory: c:\Users\sbrad\Documents\Uni\In-this-deep-together\Improve\Simulate new data\The_way
Found 50 checkpoints: ['cgan_2.pt', 'cgan_4.pt', 'cgan_6.pt', 'cgan_8.pt', 'cgan_10.pt', 'cgan_12.pt', 'cgan_14.pt', 'cgan_16.pt', 'cgan_18.pt', 'cgan_20.pt', 'cgan_22.pt', 'cgan_24.pt', 'cgan_26.pt', 'cgan_28.pt', 'cgan_30.pt', 'cgan_32.pt', 'cgan_34.pt', 'cgan_36.pt', 'cgan_38.pt', 'cgan_40.pt', 'cgan_42.pt', 'cgan_44.pt', 'cgan_46.pt', 'cgan_48.pt', 'cgan_50.pt', 'cgan_52.pt', 'cgan_54.pt', 'cgan_56.pt', 'cgan_58.pt', 'cgan_60.pt', 'cgan_62.pt', 'cgan_64.pt', 'cgan_66.pt', 'cgan_68.pt', 'cgan_70.pt', 'cgan_72.pt', 'cgan_74.pt', 'cgan_76.pt', 'cgan_78.pt', 'cgan_80.pt', 'cgan_82.pt', 'cgan_84.pt', 'cgan_86.pt', 'cgan_88.pt', 'cgan_90.pt', 'cgan_92.pt', 'cgan_94.pt', 'cgan_96.pt', 'cgan_98.pt', 'cgan_100.pt']


C:\Users\sbrad\AppData\Local\Temp\ipykernel_7348\1401088999.py:96: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  payload = torch.load(checkpoint_path, map_location=device)


Inferred latent_dim=128, cond_dim=6
Generating features for class-fixed inputs (this may take a while)
Generating features for epoch 2 from cgan_2.pt
Generating features for epoch 4 from cgan_4.pt
Generating features for epoch 6 from cgan_6.pt
Generating features for epoch 8 from cgan_8.pt
Generating features for epoch 10 from cgan_10.pt
Generating features for epoch 12 from cgan_12.pt
Generating features for epoch 14 from cgan_14.pt
Generating features for epoch 16 from cgan_16.pt
Generating features for epoch 18 from cgan_18.pt
Generating features for epoch 20 from cgan_20.pt
Generating features for epoch 22 from cgan_22.pt
Generating features for epoch 24 from cgan_24.pt
Generating features for epoch 26 from cgan_26.pt
Generating features for epoch 28 from cgan_28.pt
Generating features for epoch 30 from cgan_30.pt
Generating features for epoch 32 from cgan_32.pt
Generating features for epoch 34 from cgan_34.pt
Generating features for epoch 36 from cgan_36.pt
Generating features for

c:\Users\sbrad\.conda\envs\soldet_reproduce\lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\sbrad\AppData\Local\Temp\ipykernel_7348\1526206134.py:171: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  img = imageio.imread(buf)


Saved classes GIF to c:\Users\sbrad\Documents\Uni\In-this-deep-together\Improve\Simulate new data\The_way\latent_classes_evolution.gif
Generating features for position-fixed inputs (this may take a while)
Generating features for epoch 2 from cgan_2.pt


C:\Users\sbrad\AppData\Local\Temp\ipykernel_7348\1401088999.py:96: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  payload = torch.load(checkpoint_path, map_location=device)


Generating features for epoch 4 from cgan_4.pt
Generating features for epoch 6 from cgan_6.pt
Generating features for epoch 8 from cgan_8.pt
Generating features for epoch 10 from cgan_10.pt
Generating features for epoch 12 from cgan_12.pt
Generating features for epoch 14 from cgan_14.pt
Generating features for epoch 16 from cgan_16.pt
Generating features for epoch 18 from cgan_18.pt
Generating features for epoch 20 from cgan_20.pt
Generating features for epoch 22 from cgan_22.pt
Generating features for epoch 24 from cgan_24.pt
Generating features for epoch 26 from cgan_26.pt
Generating features for epoch 28 from cgan_28.pt
Generating features for epoch 30 from cgan_30.pt
Generating features for epoch 32 from cgan_32.pt
Generating features for epoch 34 from cgan_34.pt
Generating features for epoch 36 from cgan_36.pt
Generating features for epoch 38 from cgan_38.pt
Generating features for epoch 40 from cgan_40.pt
Generating features for epoch 42 from cgan_42.pt
Generating features for ep